In [ ]:
!pip install -q sentence-transformers scikit-learn pyarrow pandas gdown openai

In [ ]:
import os
import gc
import json
import time
import random
import gdown
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

random.seed(42)
np.random.seed(42)

In [ ]:
PHASE2_FILE_ID = "1crcoK_1A12qbkh5ian-nwCOwgm8HV9uW"   
OPENAI_API_KEY = "..." #openai api key           

CATEGORY_NAME  = "Kindle_Store"

N_SAMPLE       = 1_000_000
N_CLUSTERS     = 100
N_PER_CLUSTER  = 8
EMBED_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_BATCH    = 512
RANDOM_SEED    = 42

OUTPUT_DIR     = Path("outputs")
REPORT_DIR     = OUTPUT_DIR / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET = OUTPUT_DIR / f"labeled_{CATEGORY_NAME}.parquet"
REPORT_FILE    = REPORT_DIR / f"hard_labeling_report_{CATEGORY_NAME}.txt"
LOCAL_PARQUET  = Path(f"cleaned2_{CATEGORY_NAME}.parquet")

In [ ]:
import zipfile

ZIP_FILE = Path("cleaned2_Kindle_Store.zip")

gdown.download(id=PHASE2_FILE_ID, output=str(ZIP_FILE), quiet=False)
with zipfile.ZipFile(ZIP_FILE, "r") as z:
  z.extractall(".")
ZIP_FILE.unlink()

Downloading...
From (original): https://drive.google.com/uc?id=1crcoK_1A12qbkh5ian-nwCOwgm8HV9uW
From (redirected): https://drive.google.com/uc?id=1crcoK_1A12qbkh5ian-nwCOwgm8HV9uW&confirm=t&uuid=6902f0da-66c1-409e-997a-bf1b1c49725b
To: /content/cleaned2_Kindle_Store.zip
100%|██████████| 380M/380M [00:03<00:00, 103MB/s]


In [ ]:
df_full = pd.read_parquet(str(LOCAL_PARQUET), columns=["parent_asin", "sentence_id", "sentence_text", "rating"])
total_rows = len(df_full)
print(f"Total sentences: {total_rows:,}")

if total_rows > N_SAMPLE:
    df = df_full.sample(n=N_SAMPLE, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Sampled        : {N_SAMPLE:,}")
else:
    df = df_full.reset_index(drop=True)

del df_full
gc.collect()

print(df.dtypes)
print(df.head(3))

Total sentences: 9,294,220
Sampled        : 1,000,000
parent_asin       object
sentence_id        int32
sentence_text     object
rating           float64
dtype: object
  parent_asin  sentence_id                                      sentence_text  \
0  B00CY3VSNO            4  mannish just makes a couple of calls and relie...   
1  B0035II952            1  john corey is a little self righteous but you ...   
2  B07CS432LZ            4  her husband wanted a divorce after 25 [GENERIC...   

   rating  
0     2.0  
1     5.0  
2     5.0  


In [ ]:
pd.set_option("display.max_colwidth", None)
print(df.head(20))


   parent_asin  sentence_id  \
0   B00CY3VSNO            4   
1   B0035II952            1   
2   B07CS432LZ            4   
3   B082J6HG7D            2   
4   B00YKT7OAQ            1   
5   B083SHBCKR            1   
6   B08DZ6V6XW            1   
7   B06Y2K5H2S            9   
8   B09Q1S35WN            1   
9   B015TBKI3S           15   
10  B0973GWHQT            3   
11  B002L4F46I            8   
12  B00AB1ZGX4            1   
13  B00XCZNXYU            1   
14  B00AV9FIXE            9   
15  B00GF4TBRI            1   
16  B01GBYTJWS           17   
17  B00BATIJRG            2   
18  B00D676SDC            2   
19  B00WL2U3KW            3   

                                                                                                                                                                                                       sentence_text  \
0                                                                                                                                   

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda"
embed_model = SentenceTransformer(EMBED_MODEL, device=device)

embeddings = embed_model.encode(
    df["sentence_text"].tolist(),
    batch_size=EMBED_BATCH,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print(f"Embeddings shape: {embeddings.shape}")

del embed_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1954 [00:00<?, ?it/s]

Embeddings shape: (1000000, 384)


In [ ]:
from sklearn.cluster import KMeans

t0 = time.time()
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=10, max_iter=300, verbose=1)
df["cluster_id"] = kmeans.fit_predict(embeddings)
print(f"Clustering done in {time.time() - t0:.1f}s")

counts = df["cluster_id"].value_counts()
print(f"Cluster size — min: {counts.min()}  max: {counts.max()}  avg: {counts.mean():.0f}")

Initialization complete
Iteration 0, inertia 1018775.0.
Iteration 1, inertia 654582.1875.
Iteration 2, inertia 641743.5.
Iteration 3, inertia 637374.375.
Iteration 4, inertia 635063.75.
Iteration 5, inertia 633381.5.
Iteration 6, inertia 632162.75.
Iteration 7, inertia 631326.375.
Iteration 8, inertia 630719.6875.
Iteration 9, inertia 630280.1875.
Iteration 10, inertia 629914.0.
Iteration 11, inertia 629597.9375.
Iteration 12, inertia 629291.75.
Iteration 13, inertia 629023.0.
Iteration 14, inertia 628824.25.
Iteration 15, inertia 628668.375.
Iteration 16, inertia 628549.0.
Iteration 17, inertia 628452.375.
Iteration 18, inertia 628364.3125.
Iteration 19, inertia 628296.9375.
Iteration 20, inertia 628233.875.
Iteration 21, inertia 628173.9375.
Iteration 22, inertia 628116.9375.
Iteration 23, inertia 628062.75.
Iteration 24, inertia 628009.125.
Iteration 25, inertia 627953.5.
Iteration 26, inertia 627903.25.
Iteration 27, inertia 627848.4375.
Iteration 28, inertia 627800.8125.
Iteration

In [ ]:
sampled_parts = []
for cid in range(N_CLUSTERS):
    part = df[df["cluster_id"] == cid]
    sampled_parts.append(part.sample(n=min(N_PER_CLUSTER, len(part)), random_state=RANDOM_SEED))

df_to_label = pd.concat(sampled_parts, ignore_index=True)
print(f"Sentences to label: {len(df_to_label)}")

Sentences to label: 800


In [ ]:
from openai import OpenAI

_api_key = os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY)

client = OpenAI(api_key=_api_key)

SYSTEM_PROMPT = """You are an ASTE (Aspect-Sentiment-Opinion Triplet Extraction) expert for e-book and Kindle product reviews.

Extract all aspect-opinion-sentiment triplets from the given sentence.

## FORMAT
Return a JSON array of lists: [[aspect, opinion, sentiment], ...]
- aspect: the aspect term (noun/noun phrase being evaluated)
- opinion: the opinion expression (include negation if present, e.g. "not good", "not bad")
- sentiment: 0 (negative), 1 (neutral), 2 (positive)

## RULES
- If no aspect-opinion pair found, return: []
- Keep negation as part of the opinion (e.g. "not good" → sentiment 0, "not bad" → sentiment 2)
- If multiple aspects share the same opinion, create one triplet per aspect
- If one aspect has multiple opinions, create one triplet per opinion
- Implicit opinions (verbs/phrases) are valid: "I love X" → [["X", "love", 2]]
- Sentences with no specific aspect (e.g. "Very good", "I love it", "Absolutely terrible") → []
- Return ONLY the JSON array, no explanation, no markdown

## EXAMPLES
The story is engaging. → [["story", "engaging", 2]]
The ending is predictable. → [["ending", "predictable", 0]]
The story is okay. → [["story", "okay", 1]]
The screen is not good. → [["screen", "not good", 0]]
The keyboard is not bad. → [["keyboard", "not bad", 2]]
The app is not too slow. → [["app", "not too slow", 1]]
I love the keyboard. → [["keyboard", "love", 2]]
I hate the touchpad. → [["touchpad", "hate", 0]]
The app crashes often. → [["app", "crashes often", 0]]
The story is engaging but the ending is disappointing. → [["story", "engaging", 2], ["ending", "disappointing", 0]]
The sound and bass are amazing. → [["sound", "amazing", 2], ["bass", "amazing", 2]]
The plot and characters are boring. → [["plot", "boring", 0], ["characters", "boring", 0]]
The battery is long-lasting and reliable. → [["battery", "long-lasting", 2], ["battery", "reliable", 2]]
The plot is engaging, the writing is smooth, but the ending feels rushed. → [["plot", "engaging", 2], ["writing", "smooth", 2], ["ending", "rushed", 0]]
This book is better than my old one. → [["book", "better", 2]]
Very good. → []
I love it. → []
Absolutely terrible. → []"""


def extract_triplets(sentence: str, max_retries: int = 3) -> list:
    """Gọi GPT-4o-mini để trích xuất ASTE triplets. Trả về list hoặc [] nếu lỗi."""
    for attempt in range(max_retries):
        try:
            sentence = sentence.replace("[GENERIC_NOUN]", "").replace("[DOMAIN_NOISE]", "").replace("[NEG]", "").strip()
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": sentence},
                ],
                max_tokens=512,
                temperature=0,
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            return json.loads(raw)
        except json.JSONDecodeError:
            if attempt == max_retries - 1:
                return []
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 2 ** (attempt + 2)
                print(f"  Rate limit, waiting {wait}s...")
                time.sleep(wait)
            elif attempt == max_retries - 1:
                print(f"  Error on '{sentence[:40]}...': {e}")
                return []
            else:
                time.sleep(2 ** attempt)
    return []


triplets_all = []
t_start = time.time()

for idx, row in df_to_label.iterrows():
    triplets = extract_triplets(row["sentence_text"])
    triplets_all.append(triplets)

    done = len(triplets_all)
    if done % 100 == 0:
        elapsed = time.time() - t_start
        rate = done / elapsed
        remaining = (len(df_to_label) - done) / rate if rate > 0 else 0
        print(f"  [{done}/{len(df_to_label)}]  {rate:.1f} calls/s  ~{remaining:.0f}s remaining")

df_to_label = df_to_label.copy()
df_to_label["triplets"] = triplets_all
df_to_label["category_name"] = CATEGORY_NAME

n_with = sum(1 for t in triplets_all if t)
print(f"\nLabeling done in {time.time() - t_start:.1f}s  —  {n_with}/{len(triplets_all)} sentences with triplets")

  [100/800]  1.2 calls/s  ~563s remaining
  [200/800]  1.4 calls/s  ~441s remaining
  [300/800]  1.4 calls/s  ~348s remaining
  [400/800]  1.5 calls/s  ~270s remaining
  [500/800]  1.5 calls/s  ~201s remaining
  [600/800]  1.5 calls/s  ~134s remaining
  [700/800]  1.5 calls/s  ~66s remaining
  [800/800]  1.5 calls/s  ~0s remaining

Labeling done in 547.8s  —  388/800 sentences with triplets


In [ ]:
df_out = df_to_label[["parent_asin", "sentence_id", "sentence_text", "rating", "triplets", "category_name"]].copy()
df_out["triplets"] = df_out["triplets"].apply(json.dumps, ensure_ascii=False)

df_out.to_parquet(str(OUTPUT_PARQUET), index=False, compression="gzip")
print(f"Saved: {OUTPUT_PARQUET}  ({OUTPUT_PARQUET.stat().st_size / 1e6:.2f} MB)  —  {len(df_out)} rows")
print(df_out.head(5))

Saved: outputs/labeled_Kindle_Store.parquet  (0.05 MB)  —  800 rows
  parent_asin  sentence_id  \
0  B00ODDKK3M            1   
1  B0754M923T            1   
2  B07ZQPFQJ8            1   
3  B09Z1XPQW4            8   
4  B01MYDYKD0            1   

                                                                                           sentence_text  \
0                                               if wishes came true this would be [GENERIC_NOUN] of mine   
1                                                                i like the [GENERIC_NOUN] rick looks at   
2  holy moly wow wow wow i binge read it so fast and i am just itching to read the second [GENERIC_NOUN]   
3                                                   not terrible but also disbelieve this [GENERIC_NOUN]   
4                                                                       laughed the whole [GENERIC_NOUN]   

   rating triplets category_name  
0     5.0       []  Kindle_Store  
1     5.0       []  Kindle_Store

In [ ]:
n_labeled   = len(df_out)
n_with      = (df_out["triplets"] != "[]").sum()
n_empty     = n_labeled - n_with

cluster_counts = df["cluster_id"].value_counts()

# Đếm số triplets trung bình trên các câu có triplet
total_triplets = sum(len(json.loads(t)) for t in df_out["triplets"])
avg_triplets   = total_triplets / n_with if n_with else 0
n_multi        = sum(1 for t in df_out["triplets"] if len(json.loads(t)) > 1)

# Sentiment distribution
all_triplets_flat = [
    trip
    for t_str in df_out["triplets"]
    for trip in json.loads(t_str)
]
sentiment_counts = {0: 0, 1: 0, 2: 0}
# Mới — list format: [aspect, opinion, sentiment]
for t in all_triplets_flat:
    s = t[2] if len(t) == 3 else -1
    sentiment_counts[s] = sentiment_counts.get(s, 0) + 1 + 1

report_lines = [
    f"Báo cáo Gán nhãn — {CATEGORY_NAME}",
    "",
    " 1. Lấy mẫu: ",
    f"  Tổng câu Phase 2       : {total_rows:,}",
    f"  Số câu lấy mẫu         : {len(df):,}",
    "",
    " 2. Phân cụm: ",
    f"  Số cụm                 : {N_CLUSTERS}",
    f"  Trung bình câu/cụm     : {cluster_counts.mean():.1f}",
    f"  Min / Max kích thước   : {cluster_counts.min()} / {cluster_counts.max()}",
    f"  Số câu đem gán nhãn    : {n_labeled} ({N_PER_CLUSTER} câu/cụm × {N_CLUSTERS} cụm)",
    "",
    " 3. Gán nhãn: ",
    f"  Tổng câu gán nhãn      : {n_labeled}",
    f"  Có triplet             : {n_with}  ({n_with/n_labeled*100:.1f}%)",
    f"  Không có triplet       : {n_empty}  ({n_empty/n_labeled*100:.1f}%)",
    f"  Tổng số triplet        : {total_triplets}",
    f"  Trung bình triplet/câu : {avg_triplets:.2f}",
    f"  Có > 1 triplet          : {n_multi}  ({n_multi/n_labeled*100:.1f}%)",
    "",
    " 4. Phân bố Sentiment: ",
    f"  Tiêu cực (0) : {sentiment_counts.get(0,0)}",
    f"  Trung tính (1): {sentiment_counts.get(1,0)}",
    f"  Tích cực (2) : {sentiment_counts.get(2,0)}",
    "",
    " 5. File đầu ra: ",
    f"  File : {OUTPUT_PARQUET}",
    f"  Kích thước : {OUTPUT_PARQUET.stat().st_size / 1e6:.2f} MB",
    f"  Số dòng    : {n_labeled}",
    "",
    " 6. Mẫu câu đã gán nhãn: ",
]

for _, row in df_out[df_out["triplets"] != "[]"].head(10).iterrows():
    report_lines.append(f"  [{row['rating']}★] {row['sentence_text'][:90]}")
    report_lines.append(f"      triplets: {row['triplets']}")
    report_lines.append("")

report_text = "\n".join(report_lines)
REPORT_FILE.write_text(report_text, encoding="utf-8")

print(report_text)

Báo cáo Gán nhãn — Kindle_Store

 1. Lấy mẫu: 
  Tổng câu Phase 2       : 9,294,220
  Số câu lấy mẫu         : 1,000,000

 2. Phân cụm: 
  Số cụm                 : 100
  Trung bình câu/cụm     : 10000.0
  Min / Max kích thước   : 1136 / 19909
  Số câu đem gán nhãn    : 800 (8 câu/cụm × 100 cụm)

 3. Gán nhãn: 
  Tổng câu gán nhãn      : 800
  Có triplet             : 388  (48.5%)
  Không có triplet       : 412  (51.5%)
  Tổng số triplet        : 809
  Trung bình triplet/câu : 2.09
  Có > 1 triplet          : 244  (30.5%)

 4. Phân bố Sentiment: 
  Tiêu cực (0) : 380
  Trung tính (1): 100
  Tích cực (2) : 1138

 5. File đầu ra: 
  File : outputs/labeled_Kindle_Store.parquet
  Kích thước : 0.05 MB
  Số dòng    : 800

 6. Mẫu câu đã gán nhãn: 
  [5.0★] some of the descriptions of the gory [GENERIC_NOUN] were more than needed
      triplets: [["descriptions", "more than needed", 0]]

  [3.0★] good yarn [GENERIC_NOUN] really special
      triplets: [["yarn", "good", 2], ["yarn", "really spe

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PARQUET))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>